In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ==========================================
# 1. Data Acquisition
# ==========================================
def load_data():
    """Loads the Palmer Penguins dataset using seaborn."""
    df = sns.load_dataset('penguins')
    return df

# ==========================================
# 2. Preprocessing (The ML Pipeline Pattern)
# ==========================================
def preprocess_data(df):
    """Cleans, encodes, and scales the data using modular pipelines."""
    # Drop rows where the target variable is missing
    df = df.dropna(subset=['species'])
    
    # Separate features (X) and target (y)
    X = df.drop('species', axis=1)
    y = df['species']
    
    # Identify column types for different treatments
    numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
    categorical_features = ['island', 'sex']
    
    # Create a pipeline for numerical data: Impute missing values -> Scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()) # Crucial for KNN distance calculations!
    ])
    
    # Create a pipeline for categorical data: Impute missing values -> One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    # Bundle both preprocessing steps together
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    # Fit and transform the data
    X_processed = preprocessor.fit_transform(X)
    
    return X_processed, y

# ==========================================
# 3. Model Training
# ==========================================
def train_model(X_train, y_train, k=5):
    """Trains a KNN classifier."""
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    return model

# ==========================================
# 4. Evaluation
# ==========================================
def evaluate_model(model, X_test, y_test):
    """Evaluates the model and prints metrics."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test Set Accuracy: {acc:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    return y_pred


# ==========================================
# 🚀 Execute the Pipeline
# ==========================================
if __name__ == "__main__":
    # Step 1: Load
    penguins_df = load_data()
    print(f"Dataset loaded. Shape: {penguins_df.shape}\n")
    
    # Step 2: Preprocess
    X, y = preprocess_data(penguins_df)
    
    # Step 3: Split (Stratify ensures equal representation of all 3 species)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Step 4: Train
    knn_model = train_model(X_train, y_train, k=5)
    
    # Step 5: Evaluate
    evaluate_model(knn_model, X_test, y_test)

Dataset loaded. Shape: (344, 7)

Test Set Accuracy: 0.9855

Classification Report:
              precision    recall  f1-score   support

      Adelie       1.00      0.97      0.98        30
   Chinstrap       0.93      1.00      0.97        14
      Gentoo       1.00      1.00      1.00        25

    accuracy                           0.99        69
   macro avg       0.98      0.99      0.98        69
weighted avg       0.99      0.99      0.99        69

